# NYC taxi line rasterization with a custom merge

This notebook rasterizes 3.4 million yellow taxi trips from January 2025 as
straight lines between pickup and dropoff zone centroids. We compare the
built-in `sum` merge against a custom log-sum merge that compresses dynamic
range, letting mid-volume corridors stand out alongside the busiest routes.

Data source: [NYC TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import LineString

import xrspatial  # registers .xrs accessor
from xrspatial.utils import ngjit

## Load taxi zones and trip data

The TLC publishes taxi zone boundaries as a shapefile and monthly trip records
as parquet files. We download both, reproject zones to WGS 84, and read only
the columns we need from the ~100 MB parquet.

In [ ]:
import requests, zipfile, io, tempfile, os
import pyproj

# Download and extract taxi zone shapefile
resp = requests.get('https://d37ci6vzurychx.cloudfront.net/misc/taxi_zones.zip')
tmpdir = tempfile.mkdtemp()
with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
    z.extractall(tmpdir)

zones = gpd.read_file(os.path.join(tmpdir, 'taxi_zones', 'taxi_zones.shp'))

# Compute centroids in the native projected CRS (EPSG:2263, feet) where
# they're accurate, then transform to WGS 84 lon/lat for rasterization.
manhattan_proj = zones[zones.borough == 'Manhattan'].copy()
cx_proj = manhattan_proj.geometry.centroid.x.values
cy_proj = manhattan_proj.geometry.centroid.y.values

transformer = pyproj.Transformer.from_crs('EPSG:2263', 'EPSG:4326', always_xy=True)
cx_wgs, cy_wgs = transformer.transform(cx_proj, cy_proj)

zones = zones.to_crs(epsg=4326)
manhattan = zones[zones.borough == 'Manhattan'].copy()
manhattan['cx'] = cx_wgs
manhattan['cy'] = cy_wgs
centroids = manhattan.set_index('LocationID')[['cx', 'cy']]

print(f'{len(manhattan)} Manhattan taxi zones')

fig, ax = plt.subplots(figsize=(4, 8))
manhattan.plot(ax=ax, facecolor='#f0f0f0', edgecolor='#999')
ax.scatter(centroids.cx, centroids.cy, s=8, color='steelblue', zorder=5)
ax.set_title('Manhattan taxi zones + centroids')
ax.set_axis_off()
plt.tight_layout()

In [ ]:
trips = pd.read_parquet(
    'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet',
    columns=['PULocationID', 'DOLocationID', 'total_amount', 'trip_distance'],
)

# Keep Manhattan-to-Manhattan trips with positive fare and nontrivial distance
mh_ids = set(centroids.index)
mask = (
    trips.PULocationID.isin(mh_ids)
    & trips.DOLocationID.isin(mh_ids)
    & (trips.total_amount > 0)
    & (trips.trip_distance > 0.1)
    & (trips.PULocationID != trips.DOLocationID)
)
trips = trips[mask]
print(f'{len(trips):,} Manhattan-to-Manhattan trips after filtering')

## Build route lines

Group trips by pickup--dropoff zone pair and sum fares and counts, then draw a
straight line between zone centroids. This collapses millions of trips into a
few thousand unique routes, each carrying aggregate statistics as columns.

In [ ]:
routes = (
    trips
    .groupby(['PULocationID', 'DOLocationID'])
    .agg(trip_count=('total_amount', 'size'),
         total_fare=('total_amount', 'sum'))
    .reset_index()
)

# Attach pickup and dropoff centroids
routes = routes.merge(centroids, left_on='PULocationID', right_index=True)
routes = routes.rename(columns={'cx': 'pu_x', 'cy': 'pu_y'})
routes = routes.merge(centroids, left_on='DOLocationID', right_index=True)
routes = routes.rename(columns={'cx': 'do_x', 'cy': 'do_y'})

routes['geometry'] = [
    LineString([(r.pu_x, r.pu_y), (r.do_x, r.do_y)])
    for _, r in routes.iterrows()
]
gdf = gpd.GeoDataFrame(routes, geometry='geometry', crs='EPSG:4326')

print(f'{len(gdf):,} unique routes')
print(f'Trip count per route: min={gdf.trip_count.min()}, '
      f'median={gdf.trip_count.median():.0f}, max={gdf.trip_count.max()}')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 10))
manhattan.plot(ax=ax, facecolor='#f0f0f0', edgecolor='#ccc')
gdf.plot(ax=ax, linewidth=0.3, alpha=0.4, color='steelblue')
ax.set_title(f'{len(gdf):,} taxi routes across Manhattan')
ax.set_axis_off()
plt.tight_layout()

## Rasterize: built-in sum vs. custom log-sum

We create a raster grid covering Manhattan and burn route lines three ways:

1. **Trip count** (`merge='sum'` on `trip_count`): raw volume per pixel
2. **Total fare** (`merge='sum'` on `total_fare`): revenue per pixel
3. **Log-fare** (custom merge): `sum(log(1 + fare))` per pixel

The log transform compresses dynamic range. A \$1M corridor and a \$10K
corridor differ 100x in linear sum but only about 2.5x in log-sum. Cross-town
routes and side streets that are invisible in the linear panels become visible.

In [ ]:
# Manhattan bounding box with a bit of padding
bounds = (-74.025, 40.695, -73.930, 40.880)
width, height = 400, 800

def make_template(w, h, bounds):
    xmin, ymin, xmax, ymax = bounds
    px, py = (xmax - xmin) / w, (ymax - ymin) / h
    x = np.linspace(xmin + px / 2, xmax - px / 2, w)
    y = np.linspace(ymax - py / 2, ymin + py / 2, h)
    return xr.DataArray(np.zeros((h, w)), dims=['y', 'x'],
                        coords={'y': y, 'x': x})

template = make_template(width, height, bounds)

In [ ]:
# 1. Trip count -- linear sum
count_raster = template.xrs.rasterize(gdf, column='trip_count', merge='sum', fill=0)

# 2. Total fare -- linear sum
fare_raster = template.xrs.rasterize(gdf, column='total_fare', merge='sum', fill=0)

# 3. Log-fare -- custom non-linear merge
@ngjit
def log_fare_sum(pixel, props, is_first):
    """Sum log(1 + fare) instead of raw fares.

    Non-linear: compresses high values, widens the low range.
    A $1M route contributes log(1M) ~ 13.8 while a $1K route
    contributes log(1K) ~ 6.9, a 2:1 ratio instead of 1000:1.
    """
    val = np.log1p(props[0])
    if is_first:
        return val
    return pixel + val

log_raster = template.xrs.rasterize(gdf, column='total_fare', merge=log_fare_sum, fill=0)

In [ ]:
import pathlib
pathlib.Path('images').mkdir(exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 12), facecolor='black')

titles = ['Trip count (sum)', 'Total fare (sum)', 'Log-fare (custom merge)']
rasters = [count_raster, fare_raster, log_raster]

for ax, raster, title in zip(axes, rasters, titles):
    ax.set_facecolor('black')
    masked = raster.where(raster > 0)
    masked.plot.imshow(ax=ax, cmap='hot', add_colorbar=False,
                       interpolation='nearest')
    ax.set_title(title, color='white', fontsize=14, pad=10)
    ax.set_axis_off()

plt.tight_layout()
plt.savefig('images/nyc_taxi_lines_preview.png',
            bbox_inches='tight', dpi=120, facecolor='black')

The left two panels are dominated by a handful of bright corridors -- the
busiest routes running up and down the avenues. Everything else fades to
near-black.

The right panel applies `log(1 + x)` before summing, which compresses the
hundred-fold difference between heavy and light corridors down to about 2--3x.
Cross-town routes and side-street connections that are invisible in the linear
versions show up clearly. Same data, different merge function.

**When to use a non-linear merge:** any time a few features dominate the
signal and you want to see the rest of the distribution. Log-sum works well
for count-like or revenue-like data that spans several orders of magnitude.

### References

- [NYC TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)
- [Bresenham's line algorithm (Wikipedia)](https://en.wikipedia.org/wiki/Bresenham%27s_line_algorithm)
- [xrspatial.rasterize API docs](https://xarray-spatial.readthedocs.io/en/latest/reference/_autosummary/xrspatial.rasterize.html)